# RQ1 — Notebook 2: LSE Computation

**Research Question**: Which clustering method generates the best pseudo-labels for downstream classification?

For each dataset in the manifest, runs all 6 pseudo-label methods, applies Hungarian alignment,
trains a class-balanced RF, and records LSE.

**LSE = balanced_accuracy(RF on pseudo-labels) / balanced_accuracy(RF on true labels)**

Balanced accuracy is used throughout to handle class-imbalanced datasets correctly.
The fixed RF uses `class_weight='balanced'` to prevent majority-class collapse.

> ⚠️  **Fresh-start note**: If you changed MIN_CLASSES or the LSE formula since your last
> run, delete `data/meta_table/lse_checkpoint.csv` before running to force full recomputation.

**Output**: `data/meta_table/meta_training.csv`

In [1]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml
import torch

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR    = os.path.join(ROOT, 'data', 'raw')
META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST   = os.path.join(META_DIR, 'dataset_manifest.csv')
CHECKPOINT = os.path.join(META_DIR, 'lse_checkpoint.csv')
OUTPUT     = os.path.join(META_DIR, 'meta_training.csv')

sys.path.insert(0, os.path.join(ROOT, 'src'))
openml.config.cache_directory = RAW_DIR

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Paths OK')

Paths OK


In [2]:
SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}

manifest = pd.read_csv(MANIFEST)
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak in manifest: {leaked}'

print(f'Manifest: {len(manifest)} datasets')
manifest.head()

Manifest: 106 datasets


,dataset_id,name,n_instances,n_features,n_classes
0,46652,news_channel,20284,17,6
1,876,fri_c1_100_50,100,50,2
2,767,analcatdata_apnea1,475,3,2
3,44521,fabert_seed_3_nrows_2000_nclasses_10_ncols_100...,2000,100,7
4,45714,PriceRunner,35300,5,10


In [3]:
def load_and_split(dataset_id):
    ds = openml.datasets.get_dataset(
        dataset_id, download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))
    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc, test_size=0.2,
        random_state=SEED, stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [4]:
from lse import compute_lse, groundtruth_accuracy
from clustering import (
    pseudo_kmeans, pseudo_dbscan, pseudo_agglomerative,
    pseudo_gmm, pseudo_autoencoder, pseudo_dictlearn,
)
print('Modules loaded')

Modules loaded


In [5]:
METHODS = {
    'LSE_kmeans'   : pseudo_kmeans,
    'LSE_dbscan'   : pseudo_dbscan,
    'LSE_agg'      : pseudo_agglomerative,
    'LSE_gmm'      : pseudo_gmm,
    'LSE_autoenc'  : pseudo_autoencoder,
    'LSE_dictlearn': pseudo_dictlearn,
}

# Minimum balanced ground-truth RF accuracy — datasets below this threshold
# have label structure too weak for pseudo-labeling to be meaningful.
MIN_GT_ACC = 0.30

if os.path.exists(CHECKPOINT):
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['dataset_id'])
    results  = done_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} datasets already processed')
else:
    done_ids = set()
    results  = []
    print('Starting fresh')

all_diagnostics = []
total = len(manifest)

for i, row in manifest.iterrows():
    did   = int(row['dataset_id'])
    name  = row['name']
    n_cls = int(row['n_classes'])

    if did in done_ids:
        continue

    t0  = time.time()
    rec = {'dataset_id': did}

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)

        if X_tr.shape[1] == 0:
            raise ValueError('No numeric features')

        X_tr_sc, X_te_sc = scale(X_tr, X_te)
        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)

        if gt_acc < MIN_GT_ACC:
            print(f'[{i+1:3d}/{total}] SKIP  id={did}  (balanced_gt={gt_acc:.3f} < {MIN_GT_ACC})')
            done_ids.add(did)
            continue

        print(f'[{i+1:3d}/{total}] {name[:35]:35s}  balanced_gt={gt_acc:.3f}  n_cls={n_cls}')

        for col, fn in METHODS.items():
            rec[col] = float('nan')
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse, diag = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=col.replace('LSE_', ''),
                    dataset_name=name, verbose=True,
                )
                rec[col] = round(lse, 4)
                all_diagnostics.append(diag)
            except Exception as e:
                print(f'    {col} FAILED: {e}')
                all_diagnostics.append({'dataset': name,
                                         'method': col.replace('LSE_', ''),
                                         'failed': True, 'error': str(e)[:200]})

        elapsed = time.time() - t0
        valid_cols = [c for c in METHODS if not np.isnan(rec.get(c, np.nan))]
        best = max(valid_cols, key=lambda c: rec.get(c, -1)) if valid_cols else 'FAILED'
        rec['best_method'] = best.replace('LSE_', '')
        rec['gt_accuracy'] = round(gt_acc, 4)

        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
        print(f'           → best={rec["best_method"]}  ({elapsed:.1f}s)')

    except Exception as e:
        print(f'[{i+1:3d}/{total}] FAIL  id={did}  {name}  — {e}')
        rec.update({c: float('nan') for c in METHODS})
        rec['best_method'] = 'FAILED'
        rec['gt_accuracy']  = float('nan')
        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

print(f'\nDone. {len(results)} rows collected.')

Starting fresh
[  1/106] news_channel                         balanced_gt=0.411  n_cls=6
    kmeans      n_cl=6/6  bal_acc=0.205  gt_bal=0.411  lse=0.500  lift=0.158
    LSE_dbscan FAILED: bad allocation
    LSE_agg FAILED: Unable to allocate 1004. MiB for an array with shape (131649651,) and data type float64
    gmm         n_cl=6/6  bal_acc=0.239  gt_bal=0.411  lse=0.580  lift=0.294
    autoenc     n_cl=6/6  bal_acc=0.225  gt_bal=0.411  lse=0.548  lift=0.239
    dictlearn   n_cl=6/6  bal_acc=0.202  gt_bal=0.411  lse=0.491  lift=0.144
           → best=gmm  (56.9s)
[  2/106] fri_c1_100_50                        balanced_gt=0.722  n_cls=2
    kmeans      n_cl=2/2  bal_acc=0.530  gt_bal=0.722  lse=0.734  lift=0.136
    dbscan      n_cl=2/2  bal_acc=0.530  gt_bal=0.722  lse=0.734  lift=0.136
    agg         n_cl=2/2  bal_acc=0.530  gt_bal=0.722  lse=0.734  lift=0.136
    gmm         n_cl=2/2  bal_acc=0.621  gt_bal=0.722  lse=0.860  lift=0.545
    autoenc     n_cl=2/2  bal_acc=0.571  gt_

In [6]:
df = pd.DataFrame(results)
lse_cols = list(METHODS.keys())

# Drop fully-failed datasets
all_nan = df[lse_cols].isna().all(axis=1)
if all_nan.any():
    print(f'Dropping {all_nan.sum()} fully-failed datasets')
    df = df[~all_nan].reset_index(drop=True)

# Recompute best_method robustly
def _best(row):
    vals = {c: row[c] for c in lse_cols if not np.isnan(row[c])}
    return max(vals, key=vals.get).replace('LSE_', '') if vals else 'FAILED'

df['best_method'] = df.apply(_best, axis=1)
col_order = ['dataset_id'] + lse_cols + ['best_method', 'gt_accuracy']
df = df[col_order]

leaked = set(df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

df.to_csv(OUTPUT, index=False)
print(f'Saved → {OUTPUT}  shape={df.shape}')

Dropping 13 fully-failed datasets
Saved → c:\MLResearch\data\meta_table\meta_training.csv  shape=(86, 9)


In [7]:
if all_diagnostics:
    diag_df = pd.DataFrame(all_diagnostics)
    DIAG_PATH = os.path.join(META_DIR, 'diagnostics.csv')
    diag_df.to_csv(DIAG_PATH, index=False)

    print('=== Failure-mode frequency ===')
    for flag in ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_pseudo']:
        if flag in diag_df.columns:
            pct = diag_df[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%')
else:
    print('No diagnostics (all datasets loaded from checkpoint). Delete checkpoint to regenerate.')

=== Failure-mode frequency ===
  cluster_collapse             5.8%
  cluster_degenerate          15.3%
  mapping_collapse             5.8%
  matches_majority            17.8%
  rf_underfit_pseudo           0.2%


In [8]:
df = pd.read_csv(OUTPUT)

print('=== LSE descriptive statistics (balanced accuracy ratio) ===')
print(df[lse_cols].describe().round(3).to_string())

print('\n=== Best-method distribution ===')
print(df['best_method'].value_counts())

print('\n=== NaN counts per method ===')
print(df[lse_cols].isna().sum())

print('\n=== Per-dataset LSE std across methods (meta-learnability signal) ===')
lse_std = df[lse_cols].std(axis=1)
print(lse_std.describe().round(3).to_string())
print('(Mean std < 0.05 → methods too similar for meta-learner to distinguish)')

# Sanity: LSE values should be in [0, ~1.5]
for col in lse_cols:
    valid = df[col].dropna()
    assert (valid >= 0).all(), f'{col} has negative LSE'
    assert (valid <= 1.5).all(), f'{col} has LSE > 1.5, check for noise'

print('\nAll sanity checks passed.')
print('Ready for 03_method_redundancy.ipynb')

=== LSE descriptive statistics (balanced accuracy ratio) ===
       LSE_kmeans  LSE_dbscan  LSE_agg  LSE_gmm  LSE_autoenc  LSE_dictlearn
count      86.000      85.000   71.000   85.000       86.000         86.000
mean        0.714       0.582    0.691    0.723        0.716          0.643
std         0.221       0.294    0.221    0.203        0.229          0.217
min         0.212       0.106    0.232    0.266        0.252          0.223
25%         0.534       0.332    0.565    0.594        0.548          0.489
50%         0.733       0.582    0.707    0.725        0.731          0.635
75%         0.859       0.803    0.804    0.852        0.848          0.802
max         1.446       1.375    1.430    1.159        1.394          1.333

=== Best-method distribution ===
best_method
gmm          29
kmeans       26
autoenc      13
agg           7
dbscan        6
dictlearn     5
Name: count, dtype: int64

=== NaN counts per method ===
LSE_kmeans        0
LSE_dbscan        1
LSE_agg         